# Семинар 3. ARIMA, SARIMA и SARIMAX

**Содержание**

1. ARIMA на стационарных рядах
2. Автоматический подбор порядка: AutoARIMA
3. Прогноз продаж новых домов
4. SARIMAX: внешние признаки
5. Сезонная AutoARIMA
6. Ограничения прогноза
7. Приложение: Prophet


[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1TgD_38i9L_Zsm43tLgotYfbMSXZiPT1T)


In [ ]:
%pip install -q "statsforecast==2.1.1" "statsmodels==0.15.0" "prophet==1.5.0"

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import statsmodels.tsa.api as tsa
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, pacf

SEED = 42
np.random.seed(SEED)

In [ ]:
def show_series(frame, units="Усл. ед.", panels=False, split=None, x_title="Дата", interval=None):
    rows = len(frame.columns) if panels else 1
    fig = make_subplots(rows=rows, shared_xaxes=True, subplot_titles=list(frame.columns) if panels else None)
    if interval is not None:
        fig.add_scatter(x=interval.index, y=interval["mean_ci_upper"], mode="lines", line_width=0,
                        showlegend=False, hoverinfo="skip")
        fig.add_scatter(x=interval.index, y=interval["mean_ci_lower"], mode="lines", line_width=0,
                        fill="tonexty", fillcolor="rgba(31,119,180,0.18)",
                        name="95% прогнозный интервал", hoverinfo="skip")
    for row, name in enumerate(frame.columns, start=1):
        fig.add_scatter(x=frame.index, y=frame[name], name=name, mode="lines",
                        row=row if panels else 1, col=1)
    fig.update_traces(line_color="gray", opacity=0.5, selector=dict(name="Факт"))
    if split is not None:
        fig.add_vline(x=split, line_dash="dash", line_color="gray")
    fig.update_yaxes(title_text=units, zeroline=False)
    fig.update_xaxes(title_text=x_title, row=rows, col=1, zeroline=False)
    fig.update_layout(height=max(420, 210 * rows), showlegend=not panels,
                      legend=dict(orientation="h", y=-0.18), margin=dict(t=40), hovermode="x unified")
    fig.show()

In [ ]:
def show_acf_pacf(series, lags=28, lag_step=1, residuals=False):
    acf_result = acf(series, nlags=lags, alpha=0.05, bartlett_confint=not residuals, result_object=False)
    pacf_result = pacf(series, nlags=lags, alpha=0.05, method="ywm")
    fig = make_subplots(cols=2, subplot_titles=["ACF", "PACF"])
    for column, (estimates, interval) in enumerate([acf_result, pacf_result], start=1):
        lags = np.arange(1, len(estimates))
        lower = interval[1:, 0] - estimates[1:]
        upper = interval[1:, 1] - estimates[1:]
        fig.add_scatter(x=lags, y=upper, mode="lines", line_width=0, hoverinfo="skip", row=1, col=column)
        fig.add_scatter(
            x=lags, y=lower, mode="lines", line_width=0, fill="tonexty",
            fillcolor="rgba(31,119,180,0.18)",
            hoverinfo="skip", row=1, col=column,
        )
        for lag, value in zip(lags, estimates[1:]):
            fig.add_scatter(
                x=[lag, lag], y=[0, value], mode="lines",
                line=dict(color="#1f77b4", width=1.5),
                hoverinfo="skip", row=1, col=column,
            )
        fig.add_scatter(
            x=lags, y=estimates[1:], mode="markers", marker_color="#1f77b4",
            hoverinfo="x+y", row=1, col=column,
        )
    fig.update_xaxes(title_text="Лаг, шагов", dtick=lag_step, zeroline=False)
    fig.update_yaxes(title_text="Корреляция", range=[-1.05, 1.05], zeroline=False)
    fig.update_layout(showlegend=False, height=420, margin=dict(t=45))
    fig.show()

## 1. ARIMA на стационарных рядах

`order=(p,d,q)`: `p` — порядок AR, `d` — число обычных разностей, `q` — порядок MA. При `d=0` получаем ARMA.

### 1.1. ARIMA(2,0,0)

$$
y_t = 0.70y_{t-1} - 0.45y_{t-2} + \varepsilon_t,
\qquad \varepsilon_t \sim N(0,1).
$$

Ошибки независимы. Для генерации используем `tsa.ArmaProcess`.

In [ ]:
train_size = 160
test_size = 40
n = train_size + test_size
# В ar знаки коэффициентов противоположны знакам в уравнении ряда.
ar_process = tsa.ArmaProcess(ar=[1, -0.70, 0.45])
# burnin ослабляет влияние начальной предыстории.
np.random.seed(SEED)
stationary = pd.DataFrame({"AR(2)": ar_process.generate_sample(nsample=n, burnin=200)})

In [ ]:
show_series(pd.DataFrame({
    "Train": stationary["AR(2)"].iloc[:train_size],
    "Test": stationary["AR(2)"].iloc[train_size:],
}), split=train_size, x_title="Шаг")

In [ ]:
show_acf_pacf(stationary['AR(2)'].iloc[:train_size], lags=20)

[Документация ARIMA](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html)

`endog` — обучающий ряд; `order` — порядки модели; `trend="n"` — без константы и тренда.

In [ ]:
ar2 = ARIMA(
    endog=stationary["AR(2)"].iloc[:train_size],
    order=(2, 0, 0),
    trend="n",
).fit()
print(ar2.summary())

### Отчет модели

В `summary()` смотрим на коэффициенты, их p-value, информационные критерии и диагностику остатков.

In [ ]:
comparison = pd.DataFrame({
    "Факт": stationary["AR(2)"].iloc[train_size - test_size:],
    "ARIMA(2,0,0)": ar2.forecast(test_size),
})
show_series(comparison, split=train_size, x_title="Шаг")

### Диагностика остатков

`plot_diagnostics()` показывает остатки во времени, распределение, Q–Q и ACF. Q–Q сопоставляет квантили ошибок с квантилями нормального распределения.

[Документация plot_diagnostics](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMAResults.plot_diagnostics.html)

In [ ]:
ar2.plot_diagnostics(figsize=(12, 8)).tight_layout()

### 1.2. ARIMA(0,0,2)

$$
y_t = \varepsilon_t + 0.60\varepsilon_{t-1} + 0.45\varepsilon_{t-2}.
$$

In [ ]:
ma_process = tsa.ArmaProcess(ma=[1, 0.60, 0.45])
np.random.seed(SEED)
stationary["MA(2)"] = ma_process.generate_sample(nsample=n, burnin=200)
show_series(pd.DataFrame({
    "Train": stationary["MA(2)"].iloc[:train_size],
    "Test": stationary["MA(2)"].iloc[train_size:],
}), split=train_size, x_title="Шаг")

In [ ]:
show_acf_pacf(stationary['MA(2)'].iloc[:train_size], lags=12)

In [ ]:
ma2 = ARIMA(
    endog=stationary["MA(2)"].iloc[:train_size],
    order=(0, 0, 2),
    trend="n",
).fit()
print(ma2.summary())

In [ ]:
comparison = pd.DataFrame({
    "Факт": stationary["MA(2)"].iloc[train_size - test_size:],
    "ARIMA(0,0,2)": ma2.forecast(test_size),
})
show_series(comparison, split=train_size, x_title="Шаг")

### 1.3. ARIMA(1,0,1)

$$
y_t = 0.64y_{t-1} + \varepsilon_t + 0.37\varepsilon_{t-1}.
$$

In [ ]:
arma_process = tsa.ArmaProcess(ar=[1, -0.64], ma=[1, 0.37])
np.random.seed(SEED)
stationary["ARMA(1,1)"] = arma_process.generate_sample(nsample=n, burnin=200)
show_series(pd.DataFrame({
    "Train": stationary["ARMA(1,1)"].iloc[:train_size],
    "Test": stationary["ARMA(1,1)"].iloc[train_size:],
}), split=train_size, x_title="Шаг")

In [ ]:
show_acf_pacf(stationary['ARMA(1,1)'].iloc[:train_size], lags=12)

У чистой AR теоретическая PACF обрывается после `p`, у чистой MA теоретическая ACF — после `q`. Для смешанной ARMA обе функции могут затухать. Выборочные графики помогают наметить кандидатов.

In [ ]:
arma11 = ARIMA(
    endog=stationary["ARMA(1,1)"].iloc[:train_size],
    order=(1, 0, 1),
    trend="n",
).fit()
print(arma11.summary())

In [ ]:
comparison = pd.DataFrame({
    "Факт": stationary["ARMA(1,1)"].iloc[train_size - test_size:],
    "ARIMA(1,0,1)": arma11.forecast(test_size),
})
show_series(comparison, split=train_size, x_title="Шаг")

## 2. Автоматический подбор порядка: AutoARIMA

Подберем порядок для синтетического ARMA-ряда. AutoARIMA использует AICc и пошаговый поиск; `trace=True` выводит проверенные модели. Для этого ряда задаем `d=0`, отключаем сезонность и среднее.

[Документация AutoARIMA](https://nixtlaverse.nixtla.io/statsforecast/src/core/models.html#autoarima)

[Как AutoARIMA подбирает порядки — Forecasting: Principles and Practice, the Pythonic Way, §9.7](https://otexts.com/fpppy/09-arima.html#how-does-autoarima-work)

In [ ]:
from statsforecast.models import AutoARIMA
auto_train = stationary["ARMA(1,1)"].iloc[:train_size]
auto_test = stationary["ARMA(1,1)"].iloc[train_size:]
auto = AutoARIMA(
    d=0,
    seasonal=False,
    allowmean=False,
    trace=True,
).fit(auto_train.to_numpy())
p, q = auto.model_["arma"][:2]
{"Порядок AutoARIMA (p, d, q)": (int(p), 0, int(q))}

In [ ]:
pred_auto = auto.predict(h=test_size)["mean"]
comparison = pd.DataFrame({"AutoARIMA": pred_auto}, index=auto_test.index)
comparison = pd.concat([
    stationary["ARMA(1,1)"].iloc[train_size - test_size:].rename("Факт"), comparison,
], axis=1)
show_series(comparison, split=train_size, x_title="Шаг")

## 3. Прогноз продаж новых домов

Месячные продажи новых односемейных домов в США, **тыс. домов**, без сезонной корректировки. Источник — U.S. Census Bureau и HUD, ряд FRED `HSN1FNSA`. Используем данные с 2008 года до последней доступной даты.


### 3.1. Загрузка и исследование ряда

[Описание показателя](https://fred.stlouisfed.org/series/HSN1FNSA)

In [ ]:
houses_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=HSN1FNSA"
houses = pd.read_csv(houses_url, index_col=0, parse_dates=True)["HSN1FNSA"]
houses = houses.loc["2008":].asfreq("MS")
show_series(houses.to_frame("Продажи домов"), units="Тысяч домов в месяц")
{
    "Начало": houses.index[0],
    "Последний месяц": houses.index[-1],
    "Наблюдений": len(houses),
    "Пропусков": houses.isna().sum(),
}

Последние **24 месяца** оставляем для test, предыдущие **24 месяца** — для valid. Остальная история образует train. Порядки выбираем по train и valid; test используем для итоговой оценки.

In [ ]:
h = 24
train = houses.iloc[:-2 * h]
valid = houses.iloc[-2 * h:-h]
test = houses.iloc[-h:]
show_series(pd.DataFrame({"Train": train, "Valid": valid, "Test": test}),
            units="Тысяч домов в месяц", split=test.index[0])
{"Train": len(train), "Valid": len(valid), "Test": len(test)}

In [ ]:
monthly_profile = train.groupby(train.index.month).mean()
fig = go.Figure()
fig.add_bar(x=monthly_profile.index, y=monthly_profile)
fig.update_layout(xaxis_title="Месяц", yaxis_title="Тысяч домов", height=420, margin=dict(t=30))
fig.update_xaxes(dtick=1, zeroline=False)
fig.update_yaxes(zeroline=False)
fig.show()

### 3.2. Разность и коррелограммы

Рассмотрим обычную разность обучающего ряда: `d=1`. Построим ACF/PACF и выполним ADF-тест.

In [ ]:
from statsmodels.tsa.stattools import adfuller
houses_diff = train.diff().dropna()
show_series(houses_diff.to_frame("Обычная разность"), units="Изменение продаж, тыс. домов")
show_acf_pacf(houses_diff, lags=48, lag_step=12)
{"p-value ADF": adfuller(houses_diff)[1]}

### 3.3. Сезонная AR-часть

Начнем с одного сезонного AR-коэффициента: `P=1`, период 12. После обучения проверим остатки и прогноз на valid.

In [ ]:
houses_sar = ARIMA(
    endog=train,
    order=(0, 1, 0),
    seasonal_order=(1, 0, 0, 12),
).fit()
print(houses_sar.summary())

In [ ]:
houses_sar_errors = houses_sar.resid.iloc[houses_sar.loglikelihood_burn:]
show_acf_pacf(houses_sar_errors, lags=36, lag_step=12, residuals=True)

# Считаем все включенные AR/MA-коэффициенты, в том числе сезонные.
model_df = sum(name.startswith(("ar.", "ma.")) for name in houses_sar.param_names)
{"p-value Льюнга — Бокса": acorr_ljungbox(houses_sar_errors, lags=[24], model_df=model_df)["lb_pvalue"].iloc[0]}

In [ ]:
houses_sar_pred = houses_sar.forecast(h)
comparison = pd.DataFrame({
    "Факт": houses.loc[train.index[-24]:valid.index[-1]],
    "SARIMA(0,1,0)(1,0,0,12)": houses_sar_pred,
})
show_series(comparison, units="Тысяч домов в месяц", split=valid.index[0])
{"MAE valid, тыс. домов": round(float((valid - houses_sar_pred).abs().mean()), 2)}

### 3.4. Добавляем сезонную MA-часть

Добавим `Q=1`, сохранив `P=1`. Снова проверим остатки и прогноз на valid.

In [ ]:
houses_sarma = ARIMA(
    endog=train,
    order=(0, 1, 0),
    seasonal_order=(1, 0, 1, 12),
).fit()
print(houses_sarma.summary())

In [ ]:
houses_sarma_errors = houses_sarma.resid.iloc[houses_sarma.loglikelihood_burn:]
show_acf_pacf(houses_sarma_errors, lags=36, lag_step=12, residuals=True)
model_df = sum(name.startswith(("ar.", "ma.")) for name in houses_sarma.param_names)
{"p-value Льюнга — Бокса": acorr_ljungbox(houses_sarma_errors, lags=[24], model_df=model_df)["lb_pvalue"].iloc[0]}

In [ ]:
houses_sarma_pred = houses_sarma.forecast(h)
comparison = pd.DataFrame({
    "Факт": houses.loc[train.index[-24]:valid.index[-1]],
    "SARIMA(0,1,0)(1,0,1,12)": houses_sarma_pred,
})
show_series(comparison, units="Тысяч домов в месяц", split=valid.index[0])
{"MAE valid, тыс. домов": round(float((valid - houses_sarma_pred).abs().mean()), 2)}

### 3.5. Добавляем обычную MA-часть

Добавим `q=1`, сохранив сезонную часть модели.

In [ ]:
houses_ma = ARIMA(
    endog=train,
    order=(0, 1, 1),
    seasonal_order=(1, 0, 1, 12),
).fit()
print(houses_ma.summary())

In [ ]:
houses_ma_errors = houses_ma.resid.iloc[houses_ma.loglikelihood_burn:]
show_acf_pacf(houses_ma_errors, lags=36, lag_step=12, residuals=True)
model_df = sum(name.startswith(("ar.", "ma.")) for name in houses_ma.param_names)
{"p-value Льюнга — Бокса": acorr_ljungbox(houses_ma_errors, lags=[24], model_df=model_df)["lb_pvalue"].iloc[0]}

In [ ]:
houses_ma_pred = houses_ma.forecast(h)
comparison = pd.DataFrame({
    "Факт": houses.loc[train.index[-24]:valid.index[-1]],
    "SARIMA(0,1,1)(1,0,1,12)": houses_ma_pred,
})
show_series(comparison, units="Тысяч домов в месяц", split=valid.index[0])
{"MAE valid, тыс. домов": round(float((valid - houses_ma_pred).abs().mean()), 2)}

### 3.6. Добавляем обычную AR-часть

Добавим `p=1`, сохранив обычную MA-часть и сезонные коэффициенты.

In [ ]:
houses_arma = ARIMA(
    endog=train,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 12),
).fit()
print(houses_arma.summary())

In [ ]:
houses_arma_errors = houses_arma.resid.iloc[houses_arma.loglikelihood_burn:]
show_acf_pacf(houses_arma_errors, lags=36, lag_step=12, residuals=True)
model_df = sum(name.startswith(("ar.", "ma.")) for name in houses_arma.param_names)
{"p-value Льюнга — Бокса": acorr_ljungbox(houses_arma_errors, lags=[24], model_df=model_df)["lb_pvalue"].iloc[0]}

In [ ]:
houses_arma_pred = houses_arma.forecast(h)
comparison = pd.DataFrame({
    "Факт": houses.loc[train.index[-24]:valid.index[-1]],
    "SARIMA(1,1,1)(1,0,1,12)": houses_arma_pred,
})
show_series(comparison, units="Тысяч домов в месяц", split=valid.index[0])
{"MAE valid, тыс. домов": round(float((valid - houses_arma_pred).abs().mean()), 2)}

### 3.7. Сравнение моделей

Сравним AICc и MAE на valid. Во всех кандидатах одинаковые данные, `d=1` и `D=0`.

In [ ]:
manual_models = [houses_sar, houses_sarma, houses_ma, houses_arma]
manual_predictions = [houses_sar_pred, houses_sarma_pred, houses_ma_pred, houses_arma_pred]
pd.DataFrame({
    "AICc": [model.aicc for model in manual_models],
    "MAE valid": [float((valid - pred).abs().mean()) for pred in manual_predictions],
}, index=["SAR(1)", "+ SMA(1)", "+ обычная MA(1)", "+ обычная AR(1)"]).round(2)

### 3.8. Прогноз на test

Переобучим выбранную модель на train + valid и спрогнозируем 24 месяца. Сравним результат с сезонным наивным прогнозом.

In [ ]:
train_full = houses.iloc[:-h]
houses_final = ARIMA(
    endog=train_full,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 12),
).fit()
print(houses_final.summary())

In [ ]:
houses_test_frame = houses_final.get_forecast(h).summary_frame()
comparison = pd.DataFrame({"Факт": houses.iloc[-(h + 24):], "SARIMA": houses_test_frame["mean"]})
show_series(comparison, units="Тысяч домов в месяц", split=test.index[0], interval=houses_test_frame)
{"MAE test, тыс. домов": round(float((test - houses_test_frame["mean"]).abs().mean()), 2)}

In [ ]:
house_pred = pd.DataFrame({
    "SARIMA": houses_test_frame["mean"],
    "Сезонный наивный": np.resize(train_full.iloc[-12:], h),
})
house_pred.sub(test, axis=0).abs().mean().round(2).to_dict()

In [ ]:
houses_final.plot_diagnostics(figsize=(12, 8)).tight_layout()

## 4. SARIMAX: внешние признаки

Добавим число будних дней в месяце (Пн–Пт, без учета праздников) и индикатор марта–мая 2020 года: 1 в эти месяцы, 0 в остальные. На тестовом горизонте оба признака известны заранее.

Порядки ARIMA сохраняем. Сравним прогноз с моделью без внешних признаков.

In [ ]:
days = pd.date_range(houses.index[0], houses.index[-1] + pd.offsets.MonthEnd(), freq="B")
calendar = days.to_series().resample("MS").size().to_frame("weekdays")
calendar["covid"] = ((calendar.index >= "2020-03-01") & (calendar.index <= "2020-05-01")).astype(int)
X_train = calendar.loc[train_full.index]
X_future = calendar.loc[test.index]
X_future.head()

In [ ]:
sarimax = SARIMAX(
    endog=train_full,
    exog=X_train,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 12),
).fit(disp=False)
print(sarimax.summary())

In [ ]:
houses_sarimax_errors = sarimax.resid.iloc[sarimax.loglikelihood_burn:]
show_acf_pacf(houses_sarimax_errors, lags=36, lag_step=12, residuals=True)

model_df = sum(name.startswith(("ar.", "ma.")) for name in sarimax.param_names)
{"p-value Льюнга — Бокса": acorr_ljungbox(houses_sarimax_errors, lags=[24], model_df=model_df)["lb_pvalue"].iloc[0]}

In [ ]:
house_pred["SARIMAX"] = sarimax.forecast(h, exog=X_future)
comparison = pd.DataFrame({"Факт": houses.iloc[-(h + 24):], "SARIMAX": house_pred["SARIMAX"]})
show_series(comparison, units="Тысяч домов в месяц", split=test.index[0])
{"MAE SARIMAX, тыс. домов": round(float((test - house_pred["SARIMAX"]).abs().mean()), 2)}

## 5. Сезонная AutoARIMA

Зададим период 12 и автоматически подберем порядки. Для обучения используем ту же историю train + valid. Сравним MAE на тех же тестовых месяцах.

In [ ]:
auto_houses = AutoARIMA(
    season_length=12,
    trace=True,
).fit(train_full)
# Порядок полей: p, q, P, Q, период, d, D.
p, q, P, Q, period, d, D = auto_houses.model_["arma"]
{"order": (p, d, q), "seasonal_order": (P, D, Q, period)}

In [ ]:
auto_frame = pd.DataFrame(auto_houses.predict(h=h, level=[95]))
auto_frame.index = test.index
auto_frame = auto_frame.rename(columns={"lo-95": "mean_ci_lower", "hi-95": "mean_ci_upper"})
house_pred["AutoARIMA"] = auto_frame["mean"]
comparison = pd.DataFrame({"Факт": houses.iloc[-(h + 24):], "AutoARIMA": house_pred["AutoARIMA"]})
show_series(comparison, units="Тысяч домов в месяц", split=test.index[0], interval=auto_frame)
{"MAE AutoARIMA, тыс. домов": round(float((test - house_pred["AutoARIMA"]).abs().mean()), 2)}

In [ ]:
house_pred.sub(test, axis=0).abs().mean().sort_values().round(2).to_dict()

## 6. Ограничения прогноза

Сгенерируем положительный спрос с годовой сезонностью и небольшим снижением в последние 40 месяцев. Построим прогноз на 24 месяца и проверим его минимальное значение.

In [ ]:
t = np.arange(200)
demand = pd.Series(
    11 + 10 * np.cos(2 * np.pi * t / 12)
    - 0.02 * np.maximum(t - 160, 0)
    + np.random.normal(scale=0.05, size=len(t)),
    index=pd.date_range("2010-01-01", periods=200, freq="MS"),
    name="Спрос",
)
show_series(demand.to_frame(), units="Объем спроса, ед.")

In [ ]:
raw = ARIMA(
    endog=demand,
    order=(0, 1, 0),
    seasonal_order=(0, 1, 0, 12),
).fit()
raw_pred = raw.forecast(24)

In [ ]:
comparison = pd.DataFrame({"Факт": demand.iloc[-48:], "SARIMA": raw_pred})
show_series(comparison, units="Объем спроса, ед.", split=raw_pred.index[0])
{"Минимум прогноза": round(raw_pred.min(), 2)}

## 7. Приложение: Prophet

Применим Prophet к тому же ряду продаж домов, с теми же train + valid и test.

### 7.1. Обучение

Prophet принимает столбцы `ds` — дата и `y` — значение ряда. Зададим 95%-й прогнозный интервал.

In [ ]:
from prophet import Prophet
prophet_train = train_full.rename_axis("ds").reset_index(name="y")
prophet_model = Prophet(interval_width=0.95)
prophet_model.fit(prophet_train, seed=SEED)

### 7.2. Прогноз

Передаем даты test. `yhat` — точечный прогноз, `yhat_lower` и `yhat_upper` — границы интервала.

In [ ]:
future = pd.DataFrame({"ds": test.index})
np.random.seed(SEED)
prophet_forecast = prophet_model.predict(future).set_index("ds")
house_pred["Prophet"] = prophet_forecast["yhat"]
prophet_interval = prophet_forecast.rename(columns={
    "yhat_lower": "mean_ci_lower",
    "yhat_upper": "mean_ci_upper",
})
comparison = pd.DataFrame({"Факт": houses.iloc[-(h + 24):], "Prophet": house_pred["Prophet"]})
show_series(comparison, units="Продажи, тыс. домов", split=test.index[0], interval=prophet_interval)

### 7.3. Сравнение качества

Сравним MAE всех моделей на одних и тех же тестовых месяцах, в исходных единицах продаж.

In [ ]:
house_pred.sub(test, axis=0).abs().mean().sort_values().round(2).to_frame("MAE")